I will give a presentation on the stability of network automata in Fortaleza on CompleNet 2025.

The title that I provided is **"Small changes with large impact: robustness of life-like discrete dynamical models on complex networks"**

The abstract that I provided goes as follows:

>Cellular automata (CAs) are amongst the simplest models that can exhibit complex behaviour and even computational universality, implying that CAs may serve as a computational laboratory for the fundamental study of emergence. The emergence of complex structures in nature often appears to be robust: small changes to initial conditions generally do not qualitatively affect the global behaviour of e.g. biological organisms. Even for systems as (relatively) straightforward as CAs, however, this (in)sensitivity to initial conditions is not entirely understood. Exposing and mapping such sensitivities in complex systems is a crucial step towards better understanding, designing and exploiting robust dynamical models.
>
>Robustness and the propagation of initial defects ('damage spreading') is typically quantified by means of the Lyapunov exponent or Lyapunov profiles. Some effort has gone into mobilising this metric for various members of the five CA families, particularly for asynchronous CAs, irregular CAs, and multi-state CAs. Also members of the extended family of CAs are so-called network automata (also known as Boolean networks). Quite recently, Vispoel et al.~made substantial theoretical progress regarding the Lyapunov spectrum for Boolean networks, thereby paving the road towards a thorough statistical analysis of robustness in network automata.
>
>We continue to walk this road; in particular we investigate so-called life-like network automata (LLNAs). LLNAs are models proposed by Miranda et al.~[9] in which the famous Game of Life [10] is generalised onto a network topology. Similar to elementary CAs, such models represent some of the simplest generic dynamical processes that are nonetheless capable of generating complexity, which makes them intrinsically fascinating to study. Additionally, LLNAs are used directly as well, notably for network classification. Questions such as "which element in the chain of production may cause the most trouble?", or "which person's opinions matters the most in a social network?" naturally take shape in the context of network science. General answers to such questions can be found on a fundamental level by understanding dynamical processes like LLNAs on these networks.
>
>Concretely, we generate a large collection of network automata on scale-free networks, varying over initial conditions, wiring, and defect nodes. We perform these many calculations in a highly efficient fashion by means of a convolutional graph neural network, and generate patterns that contain the damage spread. We subsequently calculate the Lyapunov spectrum associated with each of the defect nodes, which quantifies the relative importance of small local changes on large global structures.
>
>By inspecting a number of properties that express the 'connectedness' of a particular node, we identify the character of high-impact nodes, given a particular network type and automaton dynamics. After all, it is not a priori clear how small changes will propagate: a single state change in a highly-connected cluster of nodes has a relatively small impact on that entire cluster, but has the opportunity of propagating quickly. The effect of this trade-off is not at all obvious. Our research objective therefore is to identify correlations between the Lyapunov spectrum for LLNAs on scale-free networks on the one hand, and defect nodes on the other hand, and pinpoint the 'connectedness' property that generates the highest correlation.

The focus has changed a bit, though, and that's fine.

In [ ]:
# standard preamble for the Notebooks I use
import torch as tc
import igraph as ig
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import rcParams

import matplotlib.animation as animation
from IPython.display import HTML

# Enable LaTeX and set Times New Roman as the font
rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "text.latex.preamble": r"\usepackage{amsmath}"  # Optional: Use LaTeX packages
})

from tqdm import tqdm

import sys, os
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '../..'))
sys.path.append(parent_dir)

# compact saving of data
import h5py

from src.automata import LLNA
from src.simulation import *
from src.rules import binary_indices, return_equivalent_rule, get_nonequiv_rules, return_life_like_dict
from src.networks import create_2d_torus_lattice, watts_strogatz_rewire
from src.analysis import switch_life_to_higher_node_property_values, init_config_with_dens
from src.analysis import mean_field_dens_propagation, derrida_map_analytical
from src.visual import get_ugent_cmap, get_ugent_colors_dict
from src.visual import make_animation_density_evolution, make_animation_object, add_pauses, make_igraph_animation

ugent_colors_dict = get_ugent_colors_dict()

%load_ext autoreload
%autoreload 2

## BEGINNING REMARKS

1. A lot of work has been done on "influence maximisation"
    - Mostly with probabilistic threshold models
2. We assume total knowledge of the network, the local update rule, determinism, synchronous updating, etc. Simplistic!
3. Mobilising the CA library and knowledge for network exploration
4. Different goal: not spreading (single seed), but density convergence (single defect)
    - This is a very important difference! Because now planting the seed also depends on local structure (which is not the case of the landscape is originally empty)
    - This means that we should take the neighbourhood into account, which could radically change the outcome
5. Mention that I'm happy that there are a number of specialists in the room

# 1. Context

### Impact: an intuitive example

In [ ]:
# dynamics
resolution = 9
beta = 4**2 + 5**2 + 6**2 + 7**2 + 8**2
sigma = 5**2 + 6**2 + 7**2 + 8**2
B_set = binary_indices(beta)
S_set = binary_indices(sigma)
model = LLNA(resolution, B_set, S_set, iso=True)

# Parameters
num_nodes = 50
rewiring_prob = 0.2     # Rewiring probability
avg_degree = 6
rewire_prob = 0.1

# ws_graph = ig.Graph.Watts_Strogatz(dim=1, size=num_nodes, nei=avg_degree//2, p=rewire_prob)
ws_graph.to_directed()
ws_edges = tc.tensor(ws_graph.get_edgelist()).T
ws_graph.to_undirected()

T=20
init_config = np.random.randint(0, 2, num_nodes)
configs = model.forward(ws_edges, tc.tensor(init_config[np.newaxis,:]), T=T)[0].numpy()

ugent_blue = ugent_colors_dict['ugent_blue']
ugent_red = ugent_colors_dict['re_red']
vertex_colors = np.array([[[ugent_blue, ugent_red][int(i)] for i in config] for config in configs])

interval = 1000 # ms
ani = make_igraph_animation(ws_graph, vertex_colors, interval=interval)

# show inline
HTML(ani.to_jshtml())

In [ ]:
SAVEGIF=False
fps=1

if SAVEGIF:
    savename = f"threshold_smallworld_{model.__str__()}_N{num_nodes}_T{T}.gif"

    ani = make_igraph_animation(ws_graph, vertex_colors)

    # Save animations as GIFs
    loc = './figures/gifs/'
    ani.save(loc+savename, writer='pillow', fps=fps)  # Using Pillow

### Original approach: blending research domains

In [ ]:
SAVEFIG = False

# Parameters
num_nodes = 11

random_graph = ig.Graph.Erdos_Renyi(n=num_nodes, p=0.4)

fig, ax = plt.subplots(1,1,figsize=(8,8))

# layout = lattice_graph.layout("grid")  # Grid layout
vertex_size = 35 # 200 # needs to be bigger when saving figures with high dpi
edge_width = 2
vertex_frame_width = 4

ig.plot(random_graph,
        vertex_size=vertex_size,
        vertex_color='white',
        edge_width=edge_width,
        vertex_frame_width=vertex_frame_width,
        target=ax,
        layout='circle')

if SAVEFIG:
    plt.savefig("./figures/example-random-graph.png", bbox_inches="tight", dpi=500)

# 2. The totalitarian rule

In [ ]:
# Parameters
width = 50                  # Grid size (L x L) WATCH OUT with values, becomes intensive fast. 200 takes 10 minutes
num_nodes = width**2
num_graphs = 30
rewiring_prob = 0.2         # Rewiring probability
degree = 8
num_edges = num_nodes*degree//2

# Create toroidal lattice and rewire.
# REWIRING CAN TAKE A WHILE!
lattice_graph = create_2d_torus_lattice(width, degree=degree)
small_world_graph_array = np.array([watts_strogatz_rewire(lattice_graph, rewiring_prob) for _ in tqdm(range(num_graphs), total=num_graphs)])

# find edges
small_world_edges_list = []
for small_world_graph in small_world_graph_array:
    small_world_graph.to_directed()
    small_world_edges = tc.tensor(small_world_graph.get_edgelist()).T
    small_world_edges_list.append(small_world_edges)
    small_world_graph.to_undirected()
# small_world_edges_array = np.array(small_world_edges_list)

In [ ]:
resolution=9
beta, sigma = (488,464)
B_set = binary_indices(beta)
S_set = binary_indices(sigma)
model = LLNA(resolution, B_set, S_set, iso=True)

T = 100
num_config = 10
init_dens = 0.5

configs_list = []
for small_world_edges in tqdm(small_world_edges_list, total=num_graphs):
    # run the network automaton
    init_configs = np.array([init_config_with_dens(num_nodes, init_dens) for _ in range(num_config)])
    configs = model.forward(small_world_edges, tc.tensor(init_configs), T=T).numpy().astype(int)
    # put back in L x L shape 
    configs = configs.reshape((num_config, T+1, width, width))
    configs_list.append(configs)
configs_array = np.array(configs_list)

In [ ]:
extinction_automata_per_graph = []
for idx, small_world_graph in enumerate(small_world_graph_array):
    configs_per_graph = configs_array[idx]
    configs_mask = np.mean(configs_per_graph[:,-1], axis=(1,2))==0
    extinction_automata_per_graph.append(configs_per_graph[configs_mask])

In [ ]:
# this creates a list for entries per graph.
# each entry has dimensions (max_switches, extinctions_per_graph, T+1, width, width)

kill_low_values = True

# just a single switch is OK now (we just want to visualise it)
max_switches = 1
configs_switched_list = []
for extinction_automata, small_world_graph, small_world_edges in tqdm(zip(extinction_automata_per_graph, small_world_graph_array, small_world_edges_list), total=num_graphs):
    extinctions_per_graph = extinction_automata.shape[0]
    ###############################################################
    node_property_values = np.array(small_world_graph.degree()) # np.random.permutation(num_nodes)
    ###############################################################
    if extinctions_per_graph>0:
        init_configs = extinction_automata[:,0].reshape(extinctions_per_graph, width**2)
        init_configs_switched = []
        # also run over no-switch (makes life a little easier)
        for number_of_switches in range(0,max_switches+1):
            # add them together so we can use the parallellisation of the GNN
            for init_config in init_configs:
                init_config_switched = switch_life_to_higher_node_property_values(node_property_values, init_config, number_of_switches, kill_low_values=kill_low_values)
                init_configs_switched.append(init_config_switched)
        init_configs_switched = np.array(init_configs_switched)
        configs_switched = model.forward(small_world_edges, tc.tensor(init_configs_switched), T=T).numpy().astype(int)
        configs_switched = configs_switched.reshape(max_switches+1, extinctions_per_graph, T+1, width, width)
        configs_switched_list.append(configs_switched)
    else:
        configs_switched_list = configs_switched_list + [None]

In [ ]:
# try several graph indices until there is no error message
graph_idx = 1
first_idx_with_extinction = 0
first_idx_with_full_population = 0
while True:
    config_with_extinction = configs_array[graph_idx,first_idx_with_extinction]
    if np.sum(config_with_extinction[-1])==0:
        config_switched_with_full_population = configs_switched_list[graph_idx][1,first_idx_with_full_population]
        if np.sum(config_switched_with_full_population[-1])==num_nodes:
            break
        first_idx_with_full_population += 1
    first_idx_with_extinction += 1

config_switched_with_full_population = add_pauses(config_switched_with_full_population, pause_every=T+1, pause_length=40)
config_with_extinction = add_pauses(config_with_extinction, pause_every=T+1, pause_length=40)

In [ ]:
# create and save original animation

SAVEGIF=False
loc = './figures/gifs/'
savename = f"488-464_smallworld_extinct_{model.__str__()}_{width}x{width}_T{T}.gif"
fps=10

cmap = 'Greys'
ani = make_animation_object(config_with_extinction, title=None, cmap=cmap)

if SAVEGIF:
    ani.save(loc+savename, writer='pillow', fps=fps)  # Using Pillow

# show inline
HTML(ani.to_jshtml())

In [ ]:
# create and save animation with tiny perturbation

SAVEGIF=False
loc = './figures/gifs/'
savename = f"488-464_smallworld_populated_{model.__str__()}_{width}x{width}_T{T}.gif"
fps=10

cmap = 'Greys'
ani = make_animation_object(config_switched_with_full_population, title=None, cmap=cmap)

if SAVEGIF:
    ani.save(loc+savename, writer='pillow', fps=fps)  # Using Pillow

# show inline
HTML(ani.to_jshtml())

In [ ]:
# create and save animation of the difference pattern

SAVEGIF=False
loc = './figures/gifs/'
savename = f"488-464_smallworld_difference_{model.__str__()}_{width}x{width}_T{T}.gif"
fps=10

configs_defect = np.bitwise_xor(config_with_extinction, config_switched_with_full_population)

cmap = get_ugent_cmap()
ani = make_animation_object(configs_defect, title=None, cmap=cmap)

if SAVEGIF:
    ani.save(loc+savename, writer='pillow', fps=fps)  # Using Pillow

# show inline
HTML(ani.to_jshtml())

# 4. Results: starting from structure

### State switches based on node property

In [ ]:
SAVEFIG=False
vertex_size_factor = 1
edge_width = 1
if SAVEFIG: vertex_size_factor = 10

NEWGRAPH = False

if NEWGRAPH:
    # Create a connected random graph with a specified number of nodes
    num_nodes = 8  # Number of nodes in the star network
    while True:
        random_graph = ig.Graph.Erdos_Renyi(n=num_nodes, m=int(1.5*num_nodes), directed=False)
        if random_graph.is_connected():
            break

random_config1 = 255-np.array([1, 0, 0, 1, 1, 0, 0, 1])*255

# Visualize the star graph
layout = random_graph.layout("star")
fig, ax = plt.subplots(figsize=(5, 5))
vertex_size = 15*vertex_size_factor # 150
ig.plot(random_graph, layout=layout, target=ax,vertex_size=vertex_size, vertex_color=random_config1, edge_width=edge_width)#, vertex_frame_color=ugent_colors_dict["UGent Blue"], vertex_frame_width=3)
# ax.set_title(f"$s^{{t+1}}_i = \\phi(0, 3/8)$", fontsize=fontsize, pad=-20)

if SAVEFIG:
    plt.savefig(f"./figures/random_graph_N{num_nodes}_degree-switched.png", bbox_inches="tight", dpi=500)

### Case study: influence of degree

In [ ]:
resolution = 9
beta, sigma = (488,464)
B_set = binary_indices(beta)
S_set = binary_indices(sigma)
model = LLNA(resolution, B_set, S_set, iso=True)

node_property_name_top = "degree-inverse"
node_property_name_middle = "random-shuffle"
node_property_name_bottom = "degree"

num_nodes = 50**2
degree =8

# Path to the pickle file
prefix = "final_states_after_switching_based_on_"
fileloc = "./data"

filename_top = f"{fileloc}/{prefix}{node_property_name_top}_N{num_nodes}_{model.__str__()}_degree{degree}"
filename_middle = f"{fileloc}/{prefix}{node_property_name_middle}_N{num_nodes}_{model.__str__()}_degree{degree}"
filename_bottom = f"{fileloc}/{prefix}{node_property_name_bottom}_N{num_nodes}_{model.__str__()}_degree{degree}"

npy_file_path_top = f"{filename_top}.npy"
npy_file_path_middle = f"{filename_middle}.npy"
npy_file_path_bottom = f"{filename_bottom}.npy"

# Load the npy file
y_values_array_top = np.load(npy_file_path_top)
y_values_array_middle = np.load(npy_file_path_middle)
y_values_array_bottom = np.load(npy_file_path_bottom)

y_values_arrays = [y_values_array_top, y_values_array_middle, y_values_array_bottom]

print("npy files loaded successfully.")

In [ ]:
SAVEFIG=False

scatter_colour = 'k'
bar_colour = ugent_colors_dict["ugent_blue"]
max_switches = y_values_array_top.shape[1]

fig, axs = plt.subplots(3,1,figsize=(12,7), sharex=True)
fontsize=20
alpha=0.01

for idx, (ax1, y_values_array) in enumerate(zip(axs, y_values_arrays)):
    # Create the secondary y-axis
    ax2 = ax1.twinx()

    x_vals = np.arange(y_values_array.shape[1])
    for i, y_vals in enumerate(y_values_array):
        hjitter = np.random.random(max_switches)-0.5
        x_vals_hjitter = x_vals+hjitter
        label=None
        if i==0:
            label='Final state average per sample (with jitter)'
        ax1.scatter(x_vals_hjitter, y_vals, color=scatter_colour, alpha=alpha, s=30, label=label, zorder=10)

    # Average over all samples
    y_vals_avg = np.mean(y_values_array, axis=0)  # shape: (max_switches + 1,)

    # Bin into intervals of 5
    bin_width = 5
    bins = np.arange(0, max_switches + 2, bin_width)  # +2 to include max_switches
    bin_centers = (bins[:-1] + bins[1:]) / 2

    binned_means = []
    for i in range(len(bins) - 1):
        start, end = bins[i], bins[i+1]
        bin_avg = np.mean(y_vals_avg[start:end])
        binned_means.append(bin_avg)

    # Overlay bar chart
    ax2.bar(bin_centers, binned_means, width=bin_width*0.9, color=bar_colour, alpha=1, zorder=0, label=f'Mean of final state averages (bin size ${bin_width}$)')

    # make axis pretty
    ax1.set_ylim(-0.05, 1.05)
    ax2.set_ylim(-0.05, 1.05)
    ax1.set_xlim(-2, 102)
    xticks = [0, 20, 40, 60, 80, 100]
    yticks = [0, .5, 1]
    ax1.set_xticks(xticks)
    ax1.set_xticklabels(xticks, fontsize=fontsize)
    ax1.set_yticks(yticks)
    ax2.set_yticks(yticks)
    ax1.set_yticklabels(yticks, fontsize=fontsize)
    ax2.set_yticklabels(yticks, fontsize=fontsize)

    if idx == 0:
        legend = ax1.legend(loc='upper left', bbox_to_anchor=(0,.93), fontsize=fontsize-4)
        for handle in legend.legend_handles:
            handle.set_alpha(1)
        ax2.legend(loc='upper right', bbox_to_anchor=(1,.93), fontsize=fontsize-4)

    # make sure the scatter plot is in front of the bar plot
    ax1.set_zorder(2)
    ax2.set_zorder(1)
    ax1.patch.set_visible(False)  # This is critical!

fig.supxlabel("Number of state switches", fontsize=fontsize+4)
fig.supylabel(f"State average in final time step", fontsize=fontsize+4)
# Add a right-side label manually
fig.text(1.005, 0.5, f"Binned mean of state averages", 
         fontsize=fontsize+4, 
         rotation="vertical", 
         va="center", 
         ha="center")
# ax2.set_ylabel(f"Binned mean\nof state averages", fontsize=fontsize+4)

axs[0].set_title(f"Progressively $\\mathbf{{fewer}}$ high-degree nodes are alive in initial configuration", fontsize=fontsize+4)
axs[1].set_title("Randomly chosen", fontsize=fontsize+4)
axs[2].set_title("Progressively $\\mathbf{{more}}$ high-degree nodes are alive in initial configuration", fontsize=fontsize+4)

# add horizontal line at y = 0.5 for all axes
for ax in axs:
    ax.plot([-5,105], [.5,.5], 'k:')

fig.tight_layout()

if SAVEFIG:
    plt.savefig(f"./figures/influence-of-degree-on-outcome_triple-panel.png", bbox_inches="tight", dpi=500)

### Only show some of these

In [ ]:
SAVEFIG=False

scatter_colour = 'k'
bar_colour = ugent_colors_dict["ugent_blue"]
max_switches = y_values_array_top.shape[1]

fig, axs = plt.subplots(2,1,figsize=(11,7), sharex=True)
fontsize=20
alpha=0.01

for idx, (ax1, y_values_array) in enumerate(zip(axs, y_values_arrays[1:][::-1])):

    # x_vals = np.arange(y_values_array.shape[1])
    # for i, y_vals in enumerate(y_values_array):
    #     hjitter = np.random.random(max_switches)-0.5
    #     x_vals_hjitter = x_vals+hjitter
    #     label=None
    #     if i==0:
    #         label='Final state average per sample (with jitter)'
    #     ax1.scatter(x_vals_hjitter, y_vals, color=scatter_colour, alpha=alpha, s=30, label=label, zorder=10)

    # Average over all samples
    y_vals_avg = np.mean(y_values_array, axis=0)  # shape: (max_switches + 1,)

    # Bin into intervals of 5
    bin_width = 5
    bins = np.arange(0, max_switches + 2, bin_width)  # +2 to include max_switches
    bin_centers = (bins[:-1] + bins[1:]) / 2

    binned_means = []
    for i in range(len(bins) - 1):
        start, end = bins[i], bins[i+1]
        bin_avg = np.mean(y_vals_avg[start:end])
        binned_means.append(bin_avg)

    # Overlay bar chart
    ax1.bar(bin_centers, binned_means, width=bin_width*0.9, color=bar_colour, alpha=1, zorder=0, label=f'Mean of final state averages (bin size ${bin_width}$)')

    # make axis pretty
    # ax1.set_ylim(-0.05, 1.05)
    ax1.set_ylim(-0.05, 1.05)
    ax1.set_xlim(-2, 102)
    xticks = [0, 20, 40, 60, 80, 100]
    yticks = [0, .5, 1]
    ax1.set_xticks(xticks)
    ax1.set_xticklabels(xticks, fontsize=fontsize)
    # ax1.set_yticks(yticks)
    ax1.set_yticks(yticks)
    # ax1.set_yticklabels(yticks, fontsize=fontsize)
    ax1.set_yticklabels(yticks, fontsize=fontsize)

    # if idx == 0:
    #     legend = ax1.legend(loc='upper left', bbox_to_anchor=(0,.93), fontsize=fontsize-4)
    #     for handle in legend.legend_handles:
    #         handle.set_alpha(1)
    #     ax2.legend(loc='upper right', bbox_to_anchor=(1,.93), fontsize=fontsize-4)

    # make sure the scatter plot is in front of the bar plot
    # ax1.set_zorder(2)
    # ax2.set_zorder(1)
    # ax1.patch.set_visible(False)  # This is critical!

# fig.supxlabel("Number of state switches", fontsize=fontsize+4)
axs[1].set_xlabel("Number of state switches", fontsize=fontsize+4)
fig.supylabel(f"Fraction of global final state switches", fontsize=fontsize+4)
# Add a right-side label manually
# fig.text(1.005, 0.5, f"Binned mean of state averages", 
#          fontsize=fontsize+4, 
#          rotation="vertical", 
#          va="center", 
#          ha="center")
# ax2.set_ylabel(f"Binned mean\nof state averages", fontsize=fontsize+4)

axs[0].set_title(f"Progressively $\\mathbf{{fewer}}$ high-degree nodes\nare alive in initial configuration", fontsize=fontsize+4)
axs[1].set_title("Randomly chosen", fontsize=fontsize+4)
# axs[2].set_title("Progressively $\\mathbf{{more}}$ high-degree nodes are alive in initial configuration", fontsize=fontsize+4)

# add horizontal line at y = 0.5 for all axes
for ax in axs:
    ax.plot([-5,105], [.5,.5], 'k:')

fig.tight_layout(h_pad=3.0)

if SAVEFIG:
    plt.savefig(f"./figures/influence-of-degree-on-outcome_double-panel.png", bbox_inches="tight", dpi=500)

### All structural properties

In [ ]:
resolution = 9
beta, sigma = (488,464)
B_set = binary_indices(beta)
S_set = binary_indices(sigma)
model = LLNA(resolution, B_set, S_set, iso=True)

node_properties = ["eigenvector-centrality",
                   "degree",
                   "h-index",
                   "pagerank",
                   "betweenness",
                   "closeness",
                   "average-neighbor-degrees"]

colors = [ugent_colors_dict["ugent_black"],
          ugent_colors_dict["ugent_blue"],
          ugent_colors_dict["lw_yellow"],
          ugent_colors_dict["re_red"],
          ugent_colors_dict["we_aqua"],
          ugent_colors_dict["ps_green"],
          ugent_colors_dict["di_purple"]]

linestyles = [
    '-',                # solid
    '--',               # dashed
    '-.',               # dashdot
    (0, (5, 1)),        # short dash
    (0, (5, 5)),        # long dash
    (0, (1, 1)),        # short-short (like dots, but thicker)
    (0, (3, 1, 1, 1)),  # dash-dot pattern
]

colors_dict = {node_properties[i] : colors[i] for i in range(len(node_properties))}
ls_dict = {node_properties[i] : linestyles[i] for i in range(len(node_properties))}

# Path to the pickle file
prefix = "final_states_after_switching_based_on_"
fileloc = "./data"
degree = 8
num_nodes = 50**2

y_values_array_dict = {}
for node_property_name in node_properties:
    filename = f"{fileloc}/{prefix}{node_property_name}_N{num_nodes}_{model.__str__()}_degree{degree}"
    npy_file_path = f"{filename}.npy"
    y_values_array_dict[node_property_name] = np.load(npy_file_path)

print("npy files loaded successfully.")

In [ ]:
SAVEFIG=False

from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy.interpolate import interp1d

threshold = .9

fig, ax = plt.subplots(1,1,figsize=(12,7))
fontsize=20

for key, y_values_array in y_values_array_dict.items():
    # Average over all samples
    y_vals_avg = np.mean(y_values_array, axis=0)

    # Bin into intervals of 
    bin_width = 1
    max_switches = y_vals_avg.shape[0] - 1
    bins = np.arange(0, max_switches + 2, bin_width)  # +2 to include max_switches
    bin_centers = (bins[:-1] + bins[1:]) / 2 - 0.5

    # find means in that bin
    binned_means = []
    for i in range(len(bins) - 1):
        start, end = bins[i], bins[i+1]
        bin_avg = np.mean(y_vals_avg[start:end])
        binned_means.append(bin_avg)

    # Apply LOWESS smoothing
    frac=0.3
    smoothed = lowess(binned_means, bin_centers, frac=frac)  # frac controls smoothness (0.1 = less smooth, 0.3 = more smooth)
    x_smooth = smoothed[:, 0]
    y_smooth = smoothed[:, 1]

    # Plot original and smoothed
    # ax.bar(bin_centers, binned_means, color=colors_dict[key], label='Original data', width=.9)
    ax.plot(x_smooth, y_smooth, color=colors_dict[key], linewidth=4, ls=ls_dict[key], label=key)

    # Interpolate to find when smoothed curve hits threshold
    interp = interp1d(y_smooth, x_smooth, bounds_error=False, fill_value="extrapolate")
    x_at_threshold = interp(threshold)

    print(f"The smoothed curve crosses threshold {threshold:.2f} at x ≈ {x_at_threshold:.2f}")

ax.set_ylim([0.6, 1.05])
ax.set_xlim([-1,101])
xticks = [0, 20, 40, 60, 80, 100]
yticks = [.6, .7, .8, .9, 1]
ax.set_xticks(xticks)
ax.set_xticklabels(xticks, fontsize=fontsize)
ax.set_yticks(yticks)
ax.set_yticklabels(yticks, fontsize=fontsize)
ax.set_xlabel("Number of state switches", fontsize=fontsize+4)
ax.set_ylabel("Smoothed mean of final state averages", fontsize=fontsize+4)

ax.axhline(y=threshold, color='k', linestyle=':', label=f'threshold = {threshold:.2f}', lw=1)
ax.legend(ncol=3, loc='upper left', fontsize=fontsize-4)
# ax.set_title(f'LOWESS smoothing (data fraction {frac:.2f}) and threshold detection for node property {node_property_name}', fontsize=fontsize)

fig.suptitle(f"Node state switches informed by structural properties", fontsize=fontsize+4)
fig.tight_layout()
    
if SAVEFIG:
    plt.savefig(f"./figures/influence-of-all-structural-properties-on-outcome.png", bbox_inches="tight", dpi=500)

# 4. Results: starting from structure

### Impact study by single state fixation

In [ ]:
SAVEGIF=False
loc = './figures/gifs/'
savename = f"fixed_state_random_{model.__str__()}_N{num_nodes}_T{T}.gif"
fps=1

num_nodes = 19
graph = ig.Graph.Erdos_Renyi(num_nodes, 0.3)
T = 20

configs = np.random.randint(0,2,size=(T,num_nodes))
configs[:,0] = 1

vertex_colors_over_time = np.array([
    [ugent_colors_dict["ugent_black"] if state == 1 else ugent_colors_dict["ugent_white"] for state in config] 
    for config in configs
])

ani = make_igraph_animation(graph, vertex_colors_over_time, vertex_size=25, layout=None, title=None, interval=300)

if SAVEGIF:
    ani.save(loc+savename, writer='pillow', fps=fps)  # Using Pillow

HTML(ani.to_jshtml())

### Correlation of dynamical property with structural properties

In [ ]:
num_graphs = 30
degree = 8
num_nodes = 100
num_config = 300
T=25

resolution = 9
beta, sigma = (488,464)
B_set = binary_indices(beta)
S_set = binary_indices(sigma)
model = LLNA(resolution, B_set, S_set, iso=True)

node_influences_per_graph_flat=np.load(f"data/final_state_means_after_node_forcing_{model.__str__()}_{num_graphs}smallworld_networks_degree{degree}_N{num_nodes}_{num_config}samples_T{T}.npy")
degrees = np.load(f"data/degrees_after_node_forcing_{model.__str__()}_{num_graphs}smallworld_networks_degree{degree}_N{num_nodes}_{num_config}samples_T{T}.npy")
betweenness = np.load(f"data/betweenness_after_node_forcing_{model.__str__()}_{num_graphs}smallworld_networks_degree{degree}_N{num_nodes}_{num_config}samples_T{T}.npy")
eigenvector_centrality = np.load(f"data/eigenvector_centrality_after_node_forcing_{model.__str__()}_{num_graphs}smallworld_networks_degree{degree}_N{num_nodes}_{num_config}samples_T{T}.npy")
transitivity = np.load(f"data/transitivity_after_node_forcing_{model.__str__()}_{num_graphs}smallworld_networks_degree{degree}_N{num_nodes}_{num_config}samples_T{T}.npy")

In [ ]:
SAVEFIG=False

fig, axs = plt.subplots(2,2,figsize=(12,7))
alpha=0.05
fontsize=20

# top left: eigenvector centrality
pearsonr = np.corrcoef(eigenvector_centrality, node_influences_per_graph_flat)[0,1]
axs[0,0].scatter(eigenvector_centrality, node_influences_per_graph_flat, color=colors_dict["eigenvector-centrality"], alpha=alpha, label=f"$R={pearsonr:.2f}$")
axs[0,0].set_xlabel('Eigenvector centrality', fontsize=fontsize+4)
legend=axs[0,0].legend(loc='lower right', fontsize=fontsize-4)
for handle in legend.legend_handles:
    handle.set_alpha(1)

axs[0,0].set_xlim([0.15, 1.05])
xticks = [.2, .4, .6, .8, 1]
axs[0,0].set_xticks(xticks)
axs[0,0].set_xticklabels(xticks, fontsize=fontsize)

# top right
pearsonr = np.corrcoef(degrees, node_influences_per_graph_flat)[0,1]
axs[0,1].scatter(degrees, node_influences_per_graph_flat, color=colors_dict['degree'], alpha=alpha, label=f"$R={pearsonr:.2f}$")
axs[0,1].set_xlabel('Degree', fontsize=fontsize+4)
legend=axs[0,1].legend(loc='lower right', fontsize=fontsize-4)
for handle in legend.legend_handles:
    handle.set_alpha(1)

axs[0,1].set_xlim([2.5,13.5])
xticks = [4, 6, 8, 10, 12]
axs[0,1].set_xticks(xticks)
axs[0,1].set_xticklabels(xticks, fontsize=fontsize)

# bottom left
pearsonr = np.corrcoef(betweenness, node_influences_per_graph_flat)[0,1]
axs[1,0].scatter(betweenness, node_influences_per_graph_flat, color=colors_dict['betweenness'], alpha=alpha, label=f"$R={pearsonr:.2f}$")
axs[1,0].set_xlabel('Betweenness', fontsize=fontsize+4)
legend=axs[1,0].legend(loc='lower right', fontsize=fontsize-4)
for handle in legend.legend_handles:
    handle.set_alpha(1)

axs[1,0].set_xlim([-10,310])
xticks = [0, 50, 100, 150, 200, 250, 300]
axs[1,0].set_xticks(xticks)
axs[1,0].set_xticklabels(xticks, fontsize=fontsize)

# bottom right
pearsonr = np.corrcoef(transitivity, node_influences_per_graph_flat)[0,1]
axs[1,1].scatter(transitivity, node_influences_per_graph_flat, color=ugent_colors_dict["pp_orange"], alpha=alpha, label=f"$R={pearsonr:.2f}$")
axs[1,1].set_xlabel('Clustering coefficient', fontsize=fontsize+4)
legend=axs[1,1].legend(loc='lower right', fontsize=fontsize-4)
for handle in legend.legend_handles:
    handle.set_alpha(1)

axs[1,1].set_xlim([-0.05, 1.05])
xticks= [0, .2, .4, .6, .8, 1]
axs[1,1].set_xticks(xticks)
axs[1,1].set_xticklabels(xticks, fontsize=fontsize)

# fig.suptitle(f"Data for {num_graphs} small-world networks over {num_config} samples each ($N={num_nodes}$, $T={T}$)", fontsize=fontsize+4)
fig.suptitle(f"High impact of a single fixed node correlates with structural properties", fontsize=fontsize+4)
fig.supylabel('Mean final state density', fontsize=fontsize+4)

for ax_row in axs:
    for ax in ax_row:
        yticks = [0.4, 0.5, 0.6, 0.7]
        ax.set_ylim([0.35, .75])
        ax.set_yticks(yticks)
        ax.set_yticklabels(yticks, fontsize=fontsize)

fig.tight_layout(h_pad=3.0)

if SAVEFIG:
    plt.savefig(f"figures/influence_of_forcing_node_state_vs_structure.png", bbox_inches='tight', dpi=500)

# 5. Review and outlook

This is very abstract. Why is this interesting nonetheless?
1. A internally consistent topological generalisation of the Game of Life
2. A clear example of how to use mean-field theory in practice
3. A bottom-up approach to the density classification task
4. A different take on the classical influence maximisation approach (_not_ from single seeding)
5. A well-paved road towards more dynamics-inspired network analyses
6. A clear framework for further generalisations (probabilistic, non-uniform, ...)
7. A promising tool for studying (and fighting) polarisation and extremism

However: opinion dynamics is _not_ my academic background. I hope to have a lot of inspiring conversations.

# Thank you slide

In [ ]:
from src.visual import get_heart_shaped_grid

life_like_dict = return_life_like_dict()

resolution = 9
beta, sigma = life_like_dict["replicator"]
B_set = binary_indices(beta)
S_set = binary_indices(sigma)

width = 8*12+7
T = 300
lattice_graph = create_2d_torus_lattice(width, degree=resolution-1)
lattice_graph.to_directed()
lattice_edges = tc.tensor(lattice_graph.get_edgelist()).T
lattice_graph.to_undirected()

model = LLNA(resolution, B_set, S_set, iso=True)
init_config = np.zeros((width,width))
heart = get_heart_shaped_grid()
init_config[width//2-heart.shape[0]//2:width//2+heart.shape[0]//2+1, width//2-heart.shape[1]//2:width//2+heart.shape[1]//2+1] = heart

init_config_flat = init_config.flatten()[np.newaxis,:]
configs = model.forward(lattice_edges, tc.tensor(init_config_flat), T=T)[0].numpy().astype(int)
configs = configs.reshape((T+1, width, width))

configs_with_pause = add_pauses(configs, pause_every=8, pause_length=5)

In [ ]:
SAVEGIF=False
loc = './figures/gifs/'
savename = f"animation_hearts_{model.__str__()}_{width}x{width}_T{T}.gif"

cmap = get_ugent_cmap()
ani = make_animation_object(configs_with_pause, cmap=cmap)

if SAVEGIF:
    ani.save(loc+savename, writer='pillow', fps=10)  # Using Pillow

# show inline
HTML(ani.to_jshtml())

# Information slide

### Create animated QR code

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from skimage.color import rgb2gray
from skimage.filters import threshold_otsu
from skimage.transform import resize

# Load the QR image
img_path = "./figures/QR_ORCID_rollier.png"
img = Image.open(img_path).convert("RGB")

# Convert to grayscale
gray = rgb2gray(np.array(img))

# Binarize using Otsu's threshold
thresh = threshold_otsu(gray)
binary = (gray < thresh).astype(int)  # True = black (1), False = white (0)

In [ ]:
life_like_dict = return_life_like_dict()

resolution = 9
beta, sigma = life_like_dict["day&night"]
B_set = binary_indices(beta)
S_set = binary_indices(sigma)

width = binary.shape[0]
T = 200
lattice_graph = create_2d_torus_lattice(width, degree=resolution-1)
lattice_graph.to_directed()
lattice_edges = tc.tensor(lattice_graph.get_edgelist()).T
lattice_graph.to_undirected()

model = LLNA(resolution, B_set, S_set, iso=True)

init_config_flat = binary.flatten()[np.newaxis,:]
configs = model.forward(lattice_edges, tc.tensor(init_config_flat), T=T)[0].numpy().astype(int)
configs = configs.reshape((T+1, width, width))[::-1]

In [ ]:
SAVEGIF=False
loc = './figures/gifs/'
savename = f"qr_code_{model.__str__()}_{width}x{width}_T{T}.gif"
fps=12

ani = make_animation_object(configs, cmap="Greys")

if SAVEGIF:
    ani.save(loc+savename, writer='pillow', fps=fps)  # Using Pillow

# show inline
HTML(ani.to_jshtml())

# Appendices

# A1. The small-world network

### Visualise three network types

In [ ]:
SAVEFIG = False
ugent_colors_dict = get_ugent_colors_dict()
vertex_color = ugent_colors_dict["ugent_blue"]

# Parameters
L = 4                   # Grid size (L x L) WATCH OUT with values, becomes intensive fast
num_nodes = L**2
rewiring_prob = 0.2     # Rewiring probability

# Create toroidal lattice and rewire
lattice_graph = create_2d_torus_lattice(L, degree=8)
small_world_graph = watts_strogatz_rewire(lattice_graph, rewiring_prob)
random_graph = watts_strogatz_rewire(lattice_graph, 1.)

# find edges
lattice_graph.to_directed()
lattice_edges = tc.tensor(lattice_graph.get_edgelist()).T
lattice_graph.to_undirected()

small_world_graph.to_directed()
small_world_edges = tc.tensor(small_world_graph.get_edgelist()).T
small_world_graph.to_undirected()

random_graph.to_directed()
random_edges = tc.tensor(random_graph.get_edgelist()).T
random_graph.to_undirected()

# save graphs and edges in list
graphs = [lattice_graph, small_world_graph, random_graph]
graph_names = ["Lattice", f"Small World ($p={rewiring_prob}$)", "Random"]
edges = [lattice_edges, small_world_edges, random_edges]

fig, axs = plt.subplots(1,3,figsize=(15,5))
layout = lattice_graph.layout("grid")  # Grid layout
vertex_size  = 25 # needs to be bigger when saving figures with high dpi
edge_width = 1
fontsize = 30

for ax, graph, name in zip(axs, graphs, graph_names):
    ig.plot(graph, layout=layout, vertex_size=vertex_size, vertex_color=vertex_color, edge_width=edge_width, target=ax, inline=False)
    ax.set_title(name, fontsize=fontsize)

if SAVEFIG:
    plt.savefig("./figures/four_graph_types.png", bbox_inches="tight", dpi=500)

### Effect of topology on emergent behaviour of the Game of Life

In [ ]:
# Parameters
width = 50                   # Grid size (L x L) WATCH OUT with values, becomes intensive fast
num_nodes = width**2
rewiring_prob = 0.2      # Rewiring probability

# Create toroidal lattice and rewire
lattice_graph = create_2d_torus_lattice(width, degree=8)
small_world_graph = watts_strogatz_rewire(lattice_graph, rewiring_prob)
random_graph = watts_strogatz_rewire(lattice_graph, 1.)

# find edges
lattice_graph.to_directed()
lattice_edges = tc.tensor(lattice_graph.get_edgelist()).T
lattice_graph.to_undirected()

small_world_graph.to_directed()
small_world_edges = tc.tensor(small_world_graph.get_edgelist()).T
small_world_graph.to_undirected()

random_graph.to_directed()
random_edges = tc.tensor(random_graph.get_edgelist()).T
random_graph.to_undirected()

# save graphs and edges in list
graphs = [lattice_graph, small_world_graph, random_graph]
graph_names = ["Lattice", f"Small World ($p={rewiring_prob}$)", "Random"]
edges = [lattice_edges, small_world_edges, random_edges]

# use the same init config three times
init_config = np.random.randint(0,2,size=(width,width))

In [ ]:
resolution = 9
life_like_dict = return_life_like_dict()
beta, sigma = life_like_dict["life"]
B_set = binary_indices(beta)
S_set = binary_indices(sigma)
model = LLNA(resolution, B_set, S_set, iso=True)
T = 200

idx = 1
edge = edges[idx]
graph = graphs[idx]
graph_name = graph_names[idx]

configs = model.forward(edge, tc.tensor(init_config.flatten()[np.newaxis,:]), T=T)[0].numpy().astype(int)
configs = configs.reshape((T+1,width,width))

In [ ]:
def add_pauses(grids, pause_every=8, pause_length=5):
    """
    Repeat frames at every multiple of `pause_every` to simulate a pause.

    Args:
        grids (list): List of 2D arrays representing frames.
        pause_every (int): Pause at frames divisible by this number.
        pause_length (int): How many extra times to repeat the paused frame.

    Returns:
        List of grids with pauses added.
    """
    new_grids = []
    for i, frame in enumerate(grids):
        new_grids.append(frame)
        if i % pause_every == 0:
            for _ in range(pause_length):
                new_grids.append(frame)
    return np.array(new_grids)

In [ ]:
# animations
import matplotlib.animation as animation
from IPython.display import HTML
from matplotlib import rcParams
rcParams['animation.embed_limit'] = 256  # Set a higher limit for embedding animations

def make_animation_object(grids, title=None, cmap='Greys'):
    # Set up figure
    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(grids[0], cmap=cmap)
    # change spine width
    for spine in ax.spines.values():
        spine.set_linewidth(3)
    # Update function for animation
    def update(frame):
        im.set_array(grids[frame])
        ax.set_title(title, fontsize=20)
        ax.set_xticks([]); ax.set_yticks([])
        return [im]
    # Create animation
    ani = animation.FuncAnimation(fig, update, frames=T+1, interval=100)
    plt.close()
    return ani

cmap = get_ugent_cmap()

configs_with_pause = add_pauses(configs, pause_every=T, pause_length=20)

ani = make_animation_object(configs_with_pause)
# ani = make_animation_object(defects[example_idx])

# show inline
HTML(ani.to_jshtml())

In [ ]:
SAVEGIF=False
fps=10

if SAVEGIF:
    savename = f"gol_smallworld_{model.__str__()}_{width}x{width}_T{T}.gif"

    # example_idx = np.random.randint(num_config)
    ani = make_animation_object(configs_with_pause, cmap="Greys")
    # ani = make_animation_object(defects[example_idx])

    # Save animations as GIFs
    loc = './figures/gifs/'
    ani.save(loc+savename, writer='pillow', fps=fps)  # Using Pillow

# A2. The local update rule

### Game of Life on graphs

In [ ]:
SAVEFIG = False

fig, ax = plt.subplots(1,1,figsize=(12,2.5))
fontsize=24

degree = 5

model.diagram(ax=ax, degree=degree)
ax.set_title(f"Local update rule of the Game of Life on graphs", fontsize=fontsize)
ax.set_xlabel(f"State density $\\rho$ of the node's neighbourhood", fontsize=fontsize)
ax.set_ylabel(f"Node becomes ...", fontsize=fontsize)
ax.tick_params('y', labelsize=fontsize-2)
ax.tick_params('x', labelsize=fontsize-10)

legend = ax.get_legend()
if legend:
    for text in legend.get_texts():
        text.set_fontsize(fontsize-4)  # or whatever size you want
    legend.set_title("Node was ...", prop={"size": fontsize-4})
    # legend.set_bbox_to_anchor((1, 1.05))  # Move to top-right of axes
    legend._loc = 5  # Location code: 1 = upper right

fig.tight_layout()

if SAVEFIG:
    plt.savefig(f"./figures/gol_rule_diagram_k{degree}.png", bbox_inches="tight", dpi=500)

In [ ]:
SAVEFIG=False
vertex_size_factor = 1
edge_width = 1
if SAVEFIG: vertex_size_factor = 10

# Create a star graph with a specified number of nodes
num_nodes = 9  # Number of nodes in the star network
star_graph = ig.Graph.Star(n=num_nodes)

star_config1 = 255-np.array([0, 0, 1, 1, 0, 1, 0, 0, 0])*255

# Visualize the star graph
layout = star_graph.layout("star")
fig, ax = plt.subplots(figsize=(5, 5))
vertex_size = 15*vertex_size_factor # 150
ig.plot(star_graph, layout=layout, target=ax,vertex_size=vertex_size, vertex_color=star_config1, edge_width=edge_width)#, vertex_frame_color=ugent_colors_dict["UGent Blue"], vertex_frame_width=3)
ax.set_title(f"$s^{{t+1}}_i = \\phi(0, 3/8)$", fontsize=fontsize, pad=-20)

if SAVEFIG:
    plt.savefig(f"./figures/star_graph_k{num_nodes-1}.png", bbox_inches="tight", dpi=500)

### The totalitarian update rule

In [ ]:
SAVEFIG = False

resolution = 9
beta = 488
sigma = 464
B_set = binary_indices(beta)
S_set = binary_indices(sigma)
model = LLNA(resolution, B_set, S_set, iso=True)

fig, ax = plt.subplots(1,1,figsize=(12,2.5))
fontsize=24

model.diagram(ax=ax)
ax.set_title(f"Local update rule of $\\phi^9_{{{488},{464}}}$", fontsize=fontsize)
ax.set_xlabel(f"State density $\\rho$ of the node's neighbourhood", fontsize=fontsize)
ax.set_ylabel(f"Node becomes ...", fontsize=fontsize)
ax.tick_params('y', labelsize=fontsize-2)
ax.tick_params('x', labelsize=fontsize-10)

legend = ax.get_legend()
if legend:
    for text in legend.get_texts():
        text.set_fontsize(fontsize-4)  # or whatever size you want
    legend.set_title("Node was ...", prop={"size": fontsize-4})
    # legend.set_bbox_to_anchor((1, 1.05))  # Move to top-right of axes
    legend._loc = 5  # Location code: 1 = upper right

fig.tight_layout()

if SAVEFIG:
    plt.savefig(f"./figures/phi9-488-464_rule_diagram.png", bbox_inches="tight", dpi=500)

### The mean field curve and Derrida curve of the Game of Life

In [ ]:
SAVEFIG=False

resolution=9
life_like_dict = return_life_like_dict()
(beta, sigma) = life_like_dict["life"]
B_set = binary_indices(beta)
S_set = binary_indices(sigma)
degree = 8
current_dens = np.linspace(0,1,101)

next_dens = mean_field_dens_propagation(resolution, B_set, S_set, current_dens, degree, iso=True)

fig, ax = plt.subplots(1,1,figsize=(6,6))
fontsize=20

ax.plot(current_dens, next_dens, color=ugent_colors_dict['ugent_blue'], lw=4)
ax.plot([0,1], [0,1], 'k:')
ax.set_xlim([-0.05, 1.05])
ax.set_ylim([-0.05, 1.05])
ticks = [0., .2, .4, .6, .8, 1.]
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xticklabels(ticks, fontsize=fontsize)
ax.set_yticklabels(ticks, fontsize=fontsize)
ax.set_xlabel(f"Global state average at $t=0$", fontsize=fontsize+4)
ax.set_ylabel(f"Global state average at $t=1$", fontsize=fontsize+4)

fig.suptitle(f"Mean field curve for the Game of Life", fontsize=fontsize+4)
fig.tight_layout()

if SAVEFIG:
    plt.savefig(f"figures/mean-field-curve_gol.png", bbox_inches='tight', dpi=500)
# derrida_map_analytical

In [ ]:
SAVEFIG=False

resolution=9
life_like_dict = return_life_like_dict()
(beta, sigma) = life_like_dict["life"]
B_set = binary_indices(beta)
S_set = binary_indices(sigma)
degree = 8
current_defect_dens = np.linspace(0,1,101)

next_defect_dens = derrida_map_analytical(resolution, B_set, S_set, current_defect_dens, degree, iso=True)

fig, ax = plt.subplots(1,1,figsize=(6,6))
fontsize=20

ax.plot(current_defect_dens, next_defect_dens, color=ugent_colors_dict['ugent_blue'], lw=4)
ax.plot([0,1], [0,1], 'k:')
ax.set_xlim([-0.05, 1.05])
ax.set_ylim([-0.05, 1.05])
ticks = [0., .2, .4, .6, .8, 1.]
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xticklabels(ticks, fontsize=fontsize)
ax.set_yticklabels(ticks, fontsize=fontsize)
ax.set_xlabel(f"Global defect average at $t=0$", fontsize=fontsize+4)
ax.set_ylabel(f"Global defect average at $t=1$", fontsize=fontsize+4)

fig.suptitle(f"Derrida curve for the Game of Life", fontsize=fontsize+4)
fig.tight_layout()

if SAVEFIG:
    plt.savefig(f"figures/derrida-curve_gol.png", bbox_inches='tight', dpi=500)
# derrida_map_analytical

### The mean field curve and Derrida curve of the Totalitarian rule

In [ ]:
SAVEFIG=False

resolution=9
life_like_dict = return_life_like_dict()
(beta, sigma) = (488, 464)
B_set = binary_indices(beta)
S_set = binary_indices(sigma)
degree = 8
current_dens = np.linspace(0,1,101)

next_dens = mean_field_dens_propagation(resolution, B_set, S_set, current_dens, degree, iso=True)

fig, ax = plt.subplots(1,1,figsize=(6,6))
fontsize=20

ax.plot(current_dens, next_dens, color=ugent_colors_dict['ugent_blue'], lw=4, label=f"Degree {degree}")
ax.plot([0,1], [0,1], 'k:')
ax.set_xlim([-0.05, 1.05])
ax.set_ylim([-0.05, 1.05])
ticks = [0., .2, .4, .6, .8, 1.]
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xticklabels(ticks, fontsize=fontsize)
ax.set_yticklabels(ticks, fontsize=fontsize)
ax.set_xlabel(f"Global state average at $t=0$", fontsize=fontsize+4)
ax.set_ylabel(f"Global state average at $t=1$", fontsize=fontsize+4)

ax.legend(fontsize=fontsize-4)

fig.suptitle(f"Mean field curve for the totalitarian rule", fontsize=fontsize+4)
fig.tight_layout()

if SAVEFIG:
    plt.savefig(f"figures/mean-field-curve_totalitarian.png", bbox_inches='tight', dpi=500)
# derrida_map_analytical

In [ ]:
SAVEFIG=False

resolution=9
life_like_dict = return_life_like_dict()
(beta, sigma) = (488, 464)
B_set = binary_indices(beta)
S_set = binary_indices(sigma)
degree = 8
current_defect_dens = np.linspace(0,1,101)

next_defect_dens = derrida_map_analytical(resolution, B_set, S_set, current_defect_dens, degree, iso=True)

fig, ax = plt.subplots(1,1,figsize=(6,6))
fontsize=20

ax.plot(current_defect_dens, next_defect_dens, color=ugent_colors_dict['ugent_blue'], lw=4, label=f"Degree {degree}")
ax.plot([0,1], [0,1], 'k:')
ax.set_xlim([-0.05, 1.05])
ax.set_ylim([-0.05, 1.05])
ticks = [0., .2, .4, .6, .8, 1.]
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xticklabels(ticks, fontsize=fontsize)
ax.set_yticklabels(ticks, fontsize=fontsize)
ax.set_xlabel(f"Global defect average at $t=0$", fontsize=fontsize+4)
ax.set_ylabel(f"Global defect average at $t=1$", fontsize=fontsize+4)

ax.legend(fontsize=fontsize-4)

fig.suptitle(f"Derrida curve for the totalitarian rule", fontsize=fontsize+4)
fig.tight_layout()

if SAVEFIG:
    plt.savefig(f"figures/derrida-curve_totalitarian.png", bbox_inches='tight', dpi=500)
# derrida_map_analytical

# A3. Effect of topology on emergent behaviour of the Totalitarian rule

### Density evolution from 50/50 start

In [ ]:
# Parameters
width = 50                  # Grid size (L x L) WATCH OUT with values, becomes intensive fast. 200 takes 10 minutes
num_nodes = width**2
num_graphs = 30
rewiring_prob = 0.2         # Rewiring probability
degree = 8
num_edges = num_nodes*degree//2

# Create toroidal lattice and rewire.
# REWIRING CAN TAKE A WHILE!
lattice_graph = create_2d_torus_lattice(width, degree=degree)
small_world_graph_array = np.array([watts_strogatz_rewire(lattice_graph, rewiring_prob) for _ in tqdm(range(num_graphs), total=num_graphs)])

# find edges
small_world_edges_list = []
for small_world_graph in small_world_graph_array:
    small_world_graph.to_directed()
    small_world_edges = tc.tensor(small_world_graph.get_edgelist()).T
    small_world_edges_list.append(small_world_edges)
    small_world_graph.to_undirected()
# small_world_edges_array = np.array(small_world_edges_list)

In [ ]:
T = 100
num_config = 10
init_dens = 0.5

configs_list = []
for small_world_edges in tqdm(small_world_edges_list, total=num_graphs):
    # run the network automaton
    init_configs = np.array([init_config_with_dens(num_nodes, init_dens) for _ in range(num_config)])
    configs = model.forward(small_world_edges, tc.tensor(init_configs), T=T).numpy().astype(int)
    # put back in L x L shape 
    configs = configs.reshape((num_config, T+1, width, width))
    configs_list.append(configs)
configs_array = np.array(configs_list)

In [ ]:
extinction_automata_per_graph = []
for idx, small_world_graph in enumerate(small_world_graph_array):
    configs_per_graph = configs_array[idx]
    configs_mask = np.mean(configs_per_graph[:,-1], axis=(1,2))==0
    extinction_automata_per_graph.append(configs_per_graph[configs_mask])

In [ ]:
SAVEGIF=False
loc = './figures/gifs/'
savename = f"488-464_smallworld_extinct_{model.__str__()}_{width}x{width}_T{T}.gif"
fps=10

extinction_automaton = extinction_automata_per_graph[0][0]
T = extinction_automaton.shape[0]
pause_length = 20
extinction_automaton_with_pauses = add_pauses(extinction_automaton, pause_every=T, pause_length=pause_length)

timesteps = np.concatenate(([0]*pause_length, np.arange(T)))

# plt.plot(timesteps, np.mean(extinction_automaton_with_pauses, axis=(1,2)))
cut_final_frames = 30
timesteps_ani = timesteps[:-cut_final_frames]
extinction_automaton_with_pauses_ani = extinction_automaton_with_pauses[:-cut_final_frames]

ani = make_animation_object(extinction_automaton_with_pauses[:-cut_final_frames])

if SAVEGIF:
    # Save animations as GIFs
    ani.save(loc+savename, writer='pillow', fps=fps)  # Using Pillow

# show inline
HTML(ani.to_jshtml())

In [ ]:
SAVEGIF=False
loc = './figures/gifs/'
savename = f"488-464_smallworld_extinct_plot_{model.__str__()}_{width}x{width}_T{T}.gif"
fps=10

extinction_automaton = extinction_automata_per_graph[0][0]
T = extinction_automaton.shape[0]
pause_length = 20
extinction_automaton_with_pauses = add_pauses(extinction_automaton, pause_every=T, pause_length=pause_length)

timesteps = np.concatenate(([0]*pause_length, np.arange(T)))

# plt.plot(timesteps, np.mean(extinction_automaton_with_pauses, axis=(1,2)))
cut_final_frames = 30
timesteps_ani = timesteps[:-cut_final_frames]
extinction_automaton_with_pauses_ani = extinction_automaton_with_pauses[:-cut_final_frames]

if SAVEGIF:
    # Save animations as GIFs
    ani.save(loc+savename, writer='pillow', fps=fps)  # Using Pillow

ani = make_animation_density_evolution(timesteps_ani, extinction_automaton_with_pauses_ani, vertical_color=ugent_colors_dict["ugent_blue"])

# show inline
HTML(ani.to_jshtml())

# A4. Effect of initial density and number of time steps

### Initial density

In [ ]:
SAVEFIG=False

fig, ax = plt.subplots(1,1,figsize=(12,7))
fontsize=20
alpha=0.1

modelname = 'R9B488S464'

T = 100
num_graphs = 30
num_configs_per_init_dens = 30
num_nodes = 900

filename = f"final_state_vs_initial_state_dens_rule{modelname}_T{T}_N{num_nodes}_{num_graphs}_smallworld_graphs_{num_configs_per_init_dens}_conf_per_graph.npy"
final_configs_array = np.load(f"data/{filename}")
init_dens_array = np.linspace(0,1,final_configs_array.shape[1])

# full mean
ax.plot(init_dens_array, np.mean(final_configs_array, axis=(0, 2, 3)), color=ugent_colors_dict['ugent_blue'], label='Averages', linewidth=4)

# scatter of mean of final states
init_dens_flat = init_dens_array[np.newaxis, :, np.newaxis]
init_dens_flat = np.repeat(init_dens_flat, num_graphs, axis=0)
init_dens_flat = np.repeat(init_dens_flat, num_configs_per_init_dens, axis=2)
init_dens_flat = init_dens_flat.flatten()

ax.scatter(init_dens_flat, np.mean(final_configs_array, axis=(3)).flatten(), alpha=alpha, color=ugent_colors_dict['ugent_black'], label='Individual samples', s=100)
ax.axvline(.5, color='k', ls=':', label=f"Initial density $0.5$")

ax.set_xlabel(f"Initial state density", fontsize=fontsize+4)
ax.set_ylabel(f"Final state density", fontsize=fontsize+4)

xticks = [0, .2, .4, .6, .8, 1]
yticks = [0, .2, .4, .6, .8, 1]
ax.set_xticks(xticks)
ax.set_xticklabels(xticks, fontsize=fontsize)
ax.set_yticks(yticks)
ax.set_yticklabels(yticks, fontsize=fontsize)

fig.suptitle(f"The final state of the system very strongly depends on the initial state density", fontsize=fontsize+4)

legend = ax.legend(fontsize=fontsize-4)
for handle in legend.legend_handles:
    handle.set_alpha(1)

fig.tight_layout()

if SAVEFIG:
    plt.savefig(f"final_state_vs_initial_state_dens_rule{modelname}_T{T}_N{num_nodes}_{num_graphs}_smallworld_graphs_{num_configs_per_init_dens}_conf_per_graph.png", bbox_inches='tight', dpi=500)

### Number of time steps until convergence

In [ ]:
SAVEFIG=False

modelname = 'R9B488S464'

num_graphs_per_width = 30
num_configs_per_width = 100
min_num_nodes = 50
max_num_nodes = 2000

filename = f"relaxation_time_vs_num_nodes_rule{modelname}_{num_graphs_per_width}_smallworld_graphs_{num_configs_per_width}_conf_per_graph_N{min_num_nodes}_to_{max_num_nodes}.npy"
t_array = np.load(f"data/{filename}")
num_nodes_array = np.linspace(min_num_nodes,max_num_nodes,t_array.shape[0])

fig, ax = plt.subplots(1,1,figsize=(12,7))
fontsize=20
alpha=0.5

t_array_q1 = np.quantile(t_array, 0.25, axis=1)
t_array_q2 = np.quantile(t_array, 0.5, axis=1)
t_array_q3 = np.quantile(t_array, 0.75, axis=1)

ax.plot(num_nodes_array, t_array_q2, color=ugent_colors_dict["ugent_blue"], label='Median convergence time', lw=4)
ax.fill_between(num_nodes_array, t_array_q1, t_array_q3, color=ugent_colors_dict["ugent_blue"], alpha=alpha, label="Interquartile range")

ax.set_xlabel(f"Number of nodes in the small-world network", fontsize=fontsize+4)
ax.set_ylabel(f"Average time until convergence", fontsize=fontsize+4)
ax.legend(loc='upper left', fontsize=fontsize-4)

xticks = [0, 500, 1000, 1500, 2000]
ax.set_xticks(xticks)
ax.set_xticklabels(xticks, fontsize=fontsize)
ax.set_xlim([-50, 2050])

yticks = [30, 35, 40, 45, 50]
ax.set_yticks(yticks)
ax.set_yticklabels(yticks, fontsize=fontsize)
ax.set_ylim([28, 52])

fig.suptitle(f"The larger the network, the longer it takes to reach totalitarian convergence", fontsize=fontsize+4)

fig.tight_layout()

if SAVEFIG:
    plt.savefig(f"relaxation_time_vs_num_nodes_rule{modelname}_{num_graphs_per_width}_smallworld_graphs_{num_configs_per_width}_conf_per_graph_N{min_num_nodes}_to_{max_num_nodes}.png", bbox_inches='tight', dpi=500)

In [ ]:
2000//50

np.sqrt(np.linspace(50,2000,20)).astype(int)